# AI-Driven Market Analysis for Computer Component Price Surge

**Decision Support System for Hardware Procurement in the AI Era**

Pipeline: Scraping → Preprocessing → Statistical Analysis → Sentiment → AHP-TOPSIS → Visualization.

> Run cells top to bottom. CPU runtime is sufficient.

## 1. Environment Setup

In [ ]:
import subprocess
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    REPO = "ai-era-pc-component-market-analysis"
    if Path(REPO).exists():
        subprocess.run(["git", "-C", REPO, "pull", "-q"], check=False)
    else:
        subprocess.run(
            [
                "git",
                "clone",
                "-q",
                "https://github.com/bugkey24/ai-era-pc-component-market-analysis.git",
            ],
            check=True,
        )
    get_ipython().run_line_magic("cd", REPO)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"])
    head = subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True)
    print("repo commit:", head.stdout.strip())

import nltk

nltk.download("stopwords", quiet=True)
print("Environment ready. IN_COLAB =", IN_COLAB)

## 2. Configuration & Imports

In [ ]:
from src.utils import load_config, setup_logger

CONFIG_PATH = "config.yaml"
config = load_config(CONFIG_PATH)
logger = setup_logger(name="notebook", level=config["logging"]["level"])
config["scraping"]["platforms"]

## 3. Option A — Load Existing Data

Skip live scraping (platform markup changes often). Place CSVs in `data/raw/` or use the sample generator below.

In [ ]:
from pathlib import Path

import pandas as pd

# Real experiment data ships with the repo (data/snapshot/);
# fresher local runs in data/raw take precedence, synthetic sample is the last resort.
candidates = [
    sorted(Path("data/snapshot").glob("*products*.csv")),
    sorted(Path("data/raw").glob("*products*.csv")),
]
csvs = next((c for c in candidates if c), [])

if csvs:
    data = pd.concat([pd.read_csv(f) for f in csvs], ignore_index=True)
    print(f"Loaded {len(data)} rows from {len(csvs)} file(s): {[f.name for f in csvs]}")
else:
    # Minimal sample so the pipeline is runnable end-to-end offline
    import numpy as np

    rng = np.random.default_rng(42)
    rows = []
    for cat, base, n in [("gpu", 8_000_000, 20), ("ram", 1_400_000, 20), ("ssd", 800_000, 20)]:
        for i in range(n):
            rows.append(
                {
                    "product_id": f"{cat.upper()}-{i:03d}",
                    "name": f"Sample {cat.upper()} Model {i} {rng.choice(['8GB', '16GB', '512GB', '1TB'])}",
                    "category": cat,
                    "price": f"Rp {base * rng.uniform(0.6, 1.6):,.0f}",
                    "rating": round(rng.uniform(3.8, 5.0), 1),
                    "review_count": int(rng.integers(5, 400)),
                    "seller_rating": round(rng.uniform(4.0, 5.0), 1),
                    "seller_followers": int(rng.integers(10, 5000)),
                    "source": str(rng.choice(["tokopedia", "shopee", "blibli"])),
                }
            )
    data = pd.DataFrame(rows)
    print(f"Generated {len(data)} sample rows (no snapshot or raw CSVs found)")
data.head()

## 4. Option B — Live Scraping (opt-in)

⚠️ **Expect failures from Colab.** Colab runs on Google datacenter IPs, which platforms throttle or block: Tokopedia times out, Blibli returns 403, Shopee additionally needs a Chrome install. Our validated collections ran from a **residential IP** — see `docs/09-live-experiment-results.md`. The snapshot in cell above is the reliable path.

`try_live_scrape()` below is **guarded**: a 0-row result (the expected Colab outcome) never overwrites your dataset. It stays commented out so *Run all* performs pure analysis — uncomment the last line to opt in.

In [ ]:
from src import PipelineOrchestrator  # self-contained: no earlier-cell dependency


def try_live_scrape():
    """Attempt live scraping. A 0-row result NEVER overwrites `data`."""
    pipeline = PipelineOrchestrator(CONFIG_PATH)
    scraped = pipeline._run_scraping()
    if scraped.empty:
        print(
            "Live scraping returned 0 rows (expected from datacenter IPs — "
            "see docs/10 §5). Dataset unchanged."
        )
        return None
    globals()["data"] = scraped
    print(f"Live scrape: {len(scraped)} rows — dataset replaced.")
    return scraped


# try_live_scrape()   # ← uncomment to opt in

## 5. Preprocessing & Feature Engineering

In [ ]:
from src.preprocessing import DataPreprocessor, FeatureEngineer

clean = (
    DataPreprocessor(data)
    .clean_prices()
    .handle_missing(config["preprocessing"].get("handle_missing", "drop"))
    .extract_specifications()
    .remove_outliers(threshold=config["preprocessing"].get("outlier_threshold", 3.0))
)
fe = FeatureEngineer(clean.df)
clean = fe.create_price_per_gb().create_weighted_rating().create_seller_trust_score()
df = clean.get_engineered_data()
print(f"{len(df)} rows, {len(df.columns)} columns")
df.head()

## 6. Statistical Analysis

In [ ]:
from src.analysis import StatisticalAnalyzer

stats_an = StatisticalAnalyzer(df)
summary = stats_an.describe().get_summary()
display(summary.round(2))
stats_an.correlation_matrix()
display(stats_an.price_trend_by_category())
stats_an.normality_test("price")

## 7. Sentiment Analysis

The SVM trains on the real review corpus from `data/snapshot/reviews_tokopedia.csv` (live-scraped from Tokopedia). Labels are weak supervision derived from review ratings: >=4 → positive, <=2 → negative, else neutral. When the corpus has sufficient class diversity, the model is evaluated on a held-out split; otherwise it fits the full corpus with accuracy reported as not measurable.

**Corpus stats:** 64 real reviews from 29 products across GPU/RAM/SSD. Genuine distribution: 63 positive / 1 neutral / 0 negative — Indonesian e-commerce reviews skew overwhelmingly positive, so held-out accuracy is not measurable on this corpus.

In [ ]:
import pandas as pd

from src.analysis import SentimentAnalyzer

# Real, live-scraped review corpus — no synthetic data
reviews = pd.read_csv("data/snapshot/reviews_tokopedia.csv")
ratings = pd.to_numeric(reviews["rating"], errors="coerce").fillna(3)
labels = pd.cut(
    ratings, bins=[float("-inf"), 2, 4, 5], labels=["negative", "neutral", "positive"]
).astype(str)

print(f"Real corpus: {len(reviews)} reviews from {reviews['product_id'].nunique()} products")
print("Label distribution:", labels.value_counts().to_dict())

sentiment = SentimentAnalyzer(language=config["sentiment"]["language"])
texts = reviews["review_text"].fillna("").astype(str).tolist()
try:
    sentiment.train(texts, labels.tolist())
    print("Model trained on the full real corpus (weak supervision from ratings).")
    if sentiment.accuracy is None:
        print("Held-out accuracy: not measurable — corpus lacks class diversity.")
    else:
        print(f"Held-out accuracy: {sentiment.accuracy:.2f}")
except ValueError as exc:
    print("Held-out evaluation not measurable:", exc)

## 8. AHP-TOPSIS Decision Model

In [ ]:
import numpy as np

from src.dss import AHPProcessor, TOPSISProcessor

dss_cfg = config["dss"]

ahp = AHPProcessor(dss_cfg["criteria"])
ahp.build_pairwise_matrix(dss_cfg["pairwise_matrix"])
ahp.calculate_weights().check_consistency()
print(ahp.summary())
assert ahp.is_consistent(), "Pairwise matrix inconsistent (CR >= 0.1) — revise config"

In [ ]:
# Decision matrix: map config criteria to available columns.
# Column availability differs by dataset (live cache has seller_tier but no
# seller_rating/followers; synthetic demo data is the reverse) — resolve
# with fallbacks instead of assuming.
def col_for(criterion: str) -> str:
    candidates = {
        "price": ["price"],
        "performance": ["rating"],
        "rating": ["weighted_rating", "rating"],
        "seller_reliability": ["seller_tier", "seller_trust", "rating"],
        "sentiment": ["sentiment_score", "rating"],
        "future_value": ["price_per_gb"],
    }[criterion]
    return next((c for c in candidates if c in df.columns), candidates[-1])


matrix = np.column_stack(
    [pd.to_numeric(df[col_for(c)], errors="coerce").fillna(0) for c in dss_cfg["criteria"]]
)

topsis = TOPSISProcessor(matrix, ahp.get_weights(), dss_cfg["criteria_types"])
ranking = topsis.rank()
df_ranked = (
    df.reset_index(drop=True)
    .loc[ranking["Alternative"]]
    .assign(Score=ranking["Score"].values, Rank=ranking["Rank"].values)
)
df_ranked[["name", "category", "price", "rating", "Score", "Rank"]].head(10)

## 9. Visualization

In [ ]:
import sys

# Force-reimport visualization module to bypass any stale Colab cache
for _mod in list(sys.modules):
    if "visualization" in _mod:
        del sys.modules[_mod]

from src.visualization import Visualizer

viz = Visualizer(
    df,
    output_dir="outputs/visualizations",
    show=True,  # render charts inline (figure shown before close)
    **{k: v for k, v in config["visualization"].items() if k in ("style", "palette", "dpi")},
)

print("Price Trends — all 3 categories (log scale)")
_ = viz.plot_price_trends()

print("\nCorrelation Heatmap")
_ = viz.plot_correlation_heatmap()

print("\nTOPSIS Ranking — Top 10 Products")
_ = viz.plot_ranking_bar_chart(df_ranked)

## 10. Export Results

In [ ]:
from pathlib import Path

out = Path("outputs")
out.mkdir(exist_ok=True)

# Save cleaned data
df.to_csv(out / "cleaned_data.csv", index=False)

# Save ranking
df_ranked.to_csv(out / "rankings.csv", index=False)

# Display top 10 ranked products
display_cols = ["name", "category", "price", "rating", "Score", "Rank"]
display(
    df_ranked[display_cols]
    .head(10)
    .style.format({"price": "Rp {:,.0f}", "Score": "{:.4f}"})
    .hide(axis="index")
)

# Preview cleaned dataset inline
preview_cols = [
    c for c in ["name", "category", "price", "rating", "review_count"] if c in df.columns
]
print("\ncleaned_data.csv — preview:")
display(df[preview_cols].head(10))

# Output files with sizes
print("\nOutput files:")
for p in sorted(out.iterdir()):
    if p.is_file():
        size = p.stat().st_size
    else:
        size = sum(f.stat().st_size for f in p.rglob("*") if f.is_file())
    print(f"  {p.name}  ({size / 1024:,.1f} KB)")

## 11. Conclusions

### Key Findings

| Category | Median Price (IDR) | Price Range | Observation |
| -------- | ------------------ | ----------- | ----------- |
| GPU | 9,918,000 | 2.75M – 33.5M | Highest prices driven by AI demand; widest spread |
| RAM | 8,670,000 | 2.07M – 23.6M | DDR5 premium; mid-range most popular |
| SSD | 3,388,000 | 522K – 13.1M | Best value-per-GB; NVMe adoption rising |

- **Why prices rose:** AI training/datacenter demand for GPUs; DDR5 transition premium for RAM; NVMe Gen4/5 supply constraints for SSDs.
- **Top value picks:** see ranking table above — SSDs dominate on value-per-capacity, budget GPUs score well on performance/price ratio.
- **Normalization outlook:** Base scenario (50% probability) predicts prices normalize by 2027-2028. See `docs/03-methodology.md` Phase 6.

### Methodology Notes

- Data collected live from Tokopedia via robots-guarded Apollo cache parsing (320 products, 64 reviews) on 2026-09-04.
- AHP-TOPSIS decision model uses 6 criteria: price, performance, rating, seller reliability, sentiment, and future value.
- Misclassified listings (motherboards under RAM/SSD, laptops/PC builds under GPU, accessories) are filtered by name-signature patterns; the ranking's top products are all genuine components.
- All charts render inline above and are saved to `outputs/visualizations/` for reference.

*All figures derive from the live-scraped snapshot — see `docs/09-live-experiment-results.md` for the full experiment report.*